# AI Surveillance - Suspicious Behavior Detection Demo

This notebook demonstrates **Suspicious Behavior Detection (Loitering and Running)**. Using the YOLOv11 tracker (`yolo11n.pt`), we track unique person IDs over time, monitor how long they stay in a restricted zone, and calculate their velocity between consecutive frames.

### Google Colab Instructions:
1. Click the "Runtime" menu option and select **"Change runtime type"**. Make sure **T4 GPU** is selected for optimal performance.
2. Run the cells sequentially to install dependencies, run inference, and visualize results.

## 1. Install Dependencies


In [ ]:
# Install dependencies
!pip install ultralytics opencv-python matplotlib numpy


## 2. Import Libraries


In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
import math
%matplotlib inline


## 3. Load YOLO Model


In [ ]:
print("Loading YOLO model...")
model = YOLO("yolo11n.pt")
print("YOLO model loaded successfully!")


## 4. Define Monitoring Zone


In [ ]:
# Define tracking zone parameters (normalized coordinates)
normalized_polygon = [
    [0.05, 0.40],
    [0.60, 0.40],
    [0.65, 0.95],
    [0.02, 0.95]
]


## 5. Download Test Video


In [ ]:
import urllib.request
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking-and-pause.mp4"
video_input_path = "behavior_input.mp4"
video_output_path = "behavior_output_raw.mp4"
video_final_path = "behavior_output.mp4"

if not os.path.exists(video_input_path):
    print("Downloading sample video...")
    urllib.request.urlretrieve(video_url, video_input_path)
    print("Download completed!")
else:
    print("Sample video already exists.")


## 6. Run Behavior Inference on Video File


In [ ]:
cap = cv2.VideoCapture(video_input_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Scale polygon coordinates to pixels
video_scaled_points = []
for pt in normalized_polygon:
    video_scaled_points.append([int(pt[0] * width), int(pt[1] * height)])
video_scaled_polygon = np.array(video_scaled_points, dtype=np.int32).reshape((-1, 1, 2))

# Setup Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_output_path, fourcc, fps, (width, height))

# Tracking dictionaries
track_entry_times = {}  # {track_id: first_seen_frame}
track_last_positions = {}  # {track_id: (center_x, center_y)}

print("Processing video frames...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_idx += 1
    annotated_frame = frame.copy()
    
    # Run YOLO tracking (class 0 = person)
    results = model.track(source=frame, persist=True, conf=0.25, classes=[0], imgsz=320, verbose=False)
    
    any_loitering_alert = False
    any_running_alert = False
    
    # Draw base zone background fill (default cyan/yellow-green)
    overlay = annotated_frame.copy()
    zone_color = (255, 255, 0)
    
    active_tracks = set()
    
    if results and len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            box = boxes[i]
            if box.id is None:
                continue
                
            track_id = int(box.id[0].item())
            active_tracks.add(track_id)
            
            xyxy = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, xyxy)
            
            # Center of the box
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            
            # Check if inside monitoring zone
            is_inside = cv2.pointPolygonTest(video_scaled_polygon, (float(cx), float(cy)), False) >= 0
            
            # 1. LOITERING LOGIC
            if is_inside:
                if track_id not in track_entry_times:
                    track_entry_times[track_id] = frame_idx
                
                residence_frames = frame_idx - track_entry_times[track_id]
                residence_time = residence_frames / fps
            else:
                residence_time = 0.0
                if track_id in track_entry_times:
                    del track_entry_times[track_id]
                    
            # 2. RUNNING LOGIC (speed calculation)
            speed = 0.0
            if track_id in track_last_positions:
                prev_cx, prev_cy = track_last_positions[track_id]
                dist = math.sqrt((cx - prev_cx)**2 + (cy - prev_cy)**2)
                speed = dist * fps  # Pixels per second
            track_last_positions[track_id] = (cx, cy)
            
            # Determine Alert Status
            is_running = speed > 250  # speed threshold in pixels/sec
            is_loitering = residence_time > 4.0
            
            # Color logic: Green (Safe), Yellow (Loitering Warn), Red (Loitering Alert), Orange (Running Alert)
            if is_loitering:
                box_color = (0, 0, 255) # Red
                label_tag = f"LOITERING ALERT ({residence_time:.1f}s)"
                any_loitering_alert = True
            elif residence_time > 1.5:
                box_color = (0, 255, 255) # Yellow
                label_tag = f"Loitering Warning ({residence_time:.1f}s)"
            elif is_running:
                box_color = (0, 165, 255) # Orange
                label_tag = f"RUNNING ALERT (Speed: {int(speed)}px/s)"
                any_running_alert = True
            else:
                box_color = (0, 255, 0) # Green
                label_tag = f"Normal (Speed: {int(speed)}px/s)"
                
            # Draw Bounding Box and Label
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)
            cv2.circle(annotated_frame, (cx, cy), 4, box_color, -1)
            
            label = f"ID #{track_id} | {label_tag}"
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
            cv2.rectangle(annotated_frame, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
            cv2.putText(annotated_frame, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255) if box_color in [(0, 0, 255), (0, 165, 255)] else (0, 0, 0), 1)
            
    # Clean up track_entry_times for lost tracks
    for lost_id in list(track_entry_times.keys()):
        if lost_id not in active_tracks:
            del track_entry_times[lost_id]
            
    # Apply Restricted Zone Overlay
    if any_loitering_alert:
        zone_color = (0, 0, 255)  # Red fill
    cv2.fillPoly(overlay, [video_scaled_polygon], zone_color)
    cv2.addWeighted(overlay, 0.15, annotated_frame, 0.85, 0, annotated_frame)
    cv2.polylines(annotated_frame, [video_scaled_polygon], isClosed=True, color=zone_color, thickness=2)
    
    # Draw Header Alert Banner
    if any_loitering_alert:
        cv2.rectangle(annotated_frame, (0, 0), (width, 35), (0, 0, 255), -1)
        cv2.putText(annotated_frame, "!!! SUSPICIOUS LOITERING DETECTED !!!", (width // 2 - 180, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    elif any_running_alert:
        cv2.rectangle(annotated_frame, (0, 0), (width, 35), (0, 165, 255), -1)
        cv2.putText(annotated_frame, "!!! ABNORMAL RUNNING BEHAVIOR DETECTED !!!", (width // 2 - 200, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
    out.write(annotated_frame)
    
cap.release()
out.release()
print("Video processing completed!")


## 7. Convert Output Video to H.264


In [ ]:
print("Transcoding to H.264 using FFmpeg...")
os.system(f"ffmpeg -y -i {video_output_path} -vcodec libx264 -f mp4 {video_final_path}")
print("Transcoding complete!")


## 8. Play Processed Video Inline


In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_final_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<div style="text-align: center;">
    <h3 style="color: #f39c12; font-family: sans-serif;">Suspicious Behavior Monitoring Output</h3>
    <video width="640" height="480" controls style="border: 2px solid #e67e22; border-radius: 12px; box-shadow: 0 5px 15px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
</div>
""")


## 9. Setup Colab Webcam Bridge Utilities


In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import PIL.Image

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes


## 10. Initialize Web Camera Stream CSS/JS Component


In [ ]:
def start_webcam_stream():
    js_code = """
    var video;
    var container;
    var canvas;
    var labelVal;
    
    async function initWebcam() {
      container = document.createElement('div');
      container.style.display = 'flex';
      container.style.flexDirection = 'column';
      container.style.alignItems = 'center';
      container.style.fontFamily = 'system-ui, -apple-system, sans-serif';
      container.style.background = '#18181b';
      container.style.padding = '20px';
      container.style.borderRadius = '16px';
      container.style.boxShadow = '0 10px 30px rgba(0,0,0,0.5)';
      container.style.width = '680px';
      container.style.margin = '20px auto';
      container.style.color = '#ffffff';
      container.style.border = '2px solid #27272a';
      
      const title = document.createElement('h3');
      title.innerText = 'Live Suspicious Behavior Detection Module';
      title.style.margin = '0 0 15px 0';
      title.style.color = '#f59e0b';
      title.style.fontSize = '1.3rem';
      container.appendChild(title);
      
      const videoWrapper = document.createElement('div');
      videoWrapper.style.position = 'relative';
      videoWrapper.style.width = '640px';
      videoWrapper.style.height = '480px';
      videoWrapper.style.borderRadius = '10px';
      videoWrapper.style.overflow = 'hidden';
      videoWrapper.style.border = '2px solid #3f3f46';
      
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;
      videoWrapper.appendChild(video);
      
      canvas = document.createElement('canvas');
      canvas.style.position = 'absolute';
      canvas.style.top = '0';
      canvas.style.left = '0';
      canvas.width = 640;
      canvas.height = 480;
      videoWrapper.appendChild(canvas);
      container.appendChild(videoWrapper);
      
      labelVal = document.createElement('div');
      labelVal.innerText = 'Surveillance Active - Analyzing Behavior Patterns...';
      labelVal.style.marginTop = '15px';
      labelVal.style.fontSize = '1rem';
      labelVal.style.fontWeight = 'bold';
      labelVal.style.color = '#10b981';
      container.appendChild(labelVal);
      
      const stopBtn = document.createElement('button');
      stopBtn.innerText = 'HALT BEHAVIOR MONITORING';
      stopBtn.style.marginTop = '15px';
      stopBtn.style.padding = '12px 24px';
      stopBtn.style.background = '#ef4444';
      stopBtn.style.color = 'white';
      stopBtn.style.border = 'none';
      stopBtn.style.borderRadius = '8px';
      stopBtn.style.cursor = 'pointer';
      stopBtn.style.fontWeight = 'bold';
      stopBtn.style.letterSpacing = '0.05em';
      stopBtn.style.transition = 'all 0.2s';
      stopBtn.onmouseover = () => stopBtn.style.background = '#dc2626';
      stopBtn.onmouseout = () => stopBtn.style.background = '#ef4444';
      
      stopBtn.onclick = () => {
        window.webcamActive = false;
        labelVal.innerText = 'Behavior Monitoring Stopped.';
        labelVal.style.color = '#ef4444';
        container.style.border = '2px solid #ef4444';
        const tracks = video.srcObject.getTracks();
        tracks.forEach(track => track.stop());
        video.srcObject = null;
        stopBtn.disabled = true;
        stopBtn.style.background = '#3f3f46';
        stopBtn.style.cursor = 'not-allowed';
      };
      container.appendChild(stopBtn);
      
      document.body.appendChild(container);
      
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
      video.srcObject = stream;
      await video.play();
      
      window.webcamActive = true;
    }
    
    async function captureFrame() {
      if (!window.webcamActive) return null;
      const tempCanvas = document.createElement('canvas');
      tempCanvas.width = 640;
      tempCanvas.height = 480;
      tempCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      return tempCanvas.toDataURL('image/jpeg', 0.8);
    }
    
    function drawOverlay(base64Png, statusText, alertLevel) {
      if (!window.webcamActive) return;
      const ctx = canvas.getContext('2d');
      ctx.clearRect(0, 0, 640, 480);
      
      if (base64Png) {
        const img = new Image();
        img.onload = function() {
          ctx.drawImage(img, 0, 0);
        };
        img.src = base64Png;
      }
      
      labelVal.innerText = statusText;
      if (alertLevel === 'critical') {
        labelVal.style.color = '#ef4444';
        container.style.borderColor = '#ef4444';
      } else if (alertLevel === 'warning') {
        labelVal.style.color = '#f59e0b';
        container.style.borderColor = '#f59e0b';
      } else {
        labelVal.style.color = '#10b981';
        container.style.borderColor = '#27272a';
      }
    }
    """
    display(Javascript(js_code))


## 11. Run Real-Time Webcam Behavior Stream Loop


In [ ]:
start_webcam_stream()
eval_js('initWebcam()')

height, width = 480, 640
webcam_scaled_points = []
for pt in normalized_polygon:
    webcam_scaled_points.append([int(pt[0] * width), int(pt[1] * height)])
webcam_scaled_polygon = np.array(webcam_scaled_points, dtype=np.int32).reshape((-1, 1, 2))

track_entry_frames = {}
track_last_centers = {}
frame_idx = 0
fps_est = 10  # Estimated loop speed in Colab

try:
    while True:
        js_reply = eval_js('captureFrame()')
        if js_reply is None:
            break
            
        frame = js_to_image(js_reply)
        frame_idx += 1
        
        results = model.track(source=frame, persist=True, conf=0.25, classes=[0], imgsz=320, verbose=False)
        
        overlay_array = np.zeros((480, 640, 4), dtype=np.uint8)
        active_tracks = set()
        
        any_loiter = False
        any_running = False
        
        if results and len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for i in range(len(boxes)):
                box = boxes[i]
                if box.id is None: continue
                
                track_id = int(box.id[0].item())
                active_tracks.add(track_id)
                
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, xyxy)
                
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                is_inside = cv2.pointPolygonTest(webcam_scaled_polygon, (float(cx), float(cy)), False) >= 0
                
                # Loitering logic
                if is_inside:
                    if track_id not in track_entry_frames:
                        track_entry_frames[track_id] = frame_idx
                    residence_sec = (frame_idx - track_entry_frames[track_id]) / fps_est
                else:
                    residence_sec = 0.0
                    if track_id in track_entry_frames:
                        del track_entry_frames[track_id]
                        
                # Speed logic
                speed = 0.0
                if track_id in track_last_centers:
                    prev_cx, prev_cy = track_last_centers[track_id]
                    dist = math.sqrt((cx - prev_cx)**2 + (cy - prev_cy)**2)
                    speed = dist * fps_est
                track_last_centers[track_id] = (cx, cy)
                
                is_running = speed > 220
                is_loitering = residence_sec > 4.0
                
                if is_loitering:
                    box_color = (0, 0, 255, 255) # Red
                    text_tag = f"LOITERING ALERT ({residence_sec:.1f}s)"
                    any_loiter = True
                elif residence_sec > 1.5:
                    box_color = (0, 255, 255, 255) # Yellow
                    text_tag = f"Loitering Warning ({residence_sec:.1f}s)"
                elif is_running:
                    box_color = (0, 165, 255, 255) # Orange
                    text_tag = f"RUNNING ALERT ({int(speed)}px/s)"
                    any_running = True
                else:
                    box_color = (0, 255, 0, 255) # Green
                    text_tag = f"Safe ({int(speed)}px/s)"
                    
                # Draw to overlay
                cv2.rectangle(overlay_array, (x1, y1), (x2, y2), box_color, 2)
                cv2.circle(overlay_array, (cx, cy), 4, box_color, -1)
                
                label = f"ID #{track_id} | {text_tag}"
                (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
                cv2.rectangle(overlay_array, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
                text_color = (255, 255, 255, 255) if box_color in [(0, 0, 255, 255), (0, 165, 255, 255)] else (0, 0, 0, 255)
                cv2.putText(overlay_array, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)
                
        for lost_id in list(track_entry_frames.keys()):
            if lost_id not in active_tracks:
                del track_entry_frames[lost_id]
                
        # Draw zone
        zone_color_rgba = (0, 0, 255, 38) if any_loiter else (255, 255, 0, 38)
        zone_border_rgba = (0, 0, 255, 255) if any_loiter else (255, 255, 0, 255)
        cv2.fillPoly(overlay_array, [webcam_scaled_polygon], zone_color_rgba)
        cv2.polylines(overlay_array, [webcam_scaled_polygon], True, zone_border_rgba, 2)
        
        # Warning banners
        status_msg = "Surveillance Active - Analyzing Behavior Patterns..."
        alert_level = "safe"
        
        if any_loiter:
            status_msg = "!!! SUSPICIOUS LOITERING DETECTED IN MONITORING ZONE !!!"
            alert_level = "critical"
            cv2.rectangle(overlay_array, (0, 0), (640, 35), (0, 0, 255, 255), -1)
            cv2.putText(overlay_array, status_msg, (70, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255, 255), 2)
        elif any_running:
            status_msg = "!!! SUSPICIOUS SPEED / RUNNING DETECTED !!!"
            alert_level = "warning"
            cv2.rectangle(overlay_array, (0, 0), (640, 35), (0, 165, 255, 255), -1)
            cv2.putText(overlay_array, status_msg, (120, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255, 255), 2)
            
        bbox_bytes = bbox_to_bytes(overlay_array)
        eval_js(f'drawOverlay("{bbox_bytes}", "{status_msg}", "{alert_level}")')
except Exception as e:
    print(f"Stream stopped: {e}")
